# 12 · Subset Sum, Subsets with Duplicates & Letter Combinations

More "pick / not-pick" and "choose from several options" recursion problems.

**Topics covered**
1. Subset sums (sum of **every** subset, sorted)
2. Subsets II (input has **duplicates**, answer must have no repeated subsets)
3. Letter combinations of a phone number

## 1. Subset sums

For `arr = [1, 2, 3]` there are `2³ = 8` subsets. Return the **sum of each subset**, sorted.

```
subsets : []  [1]  [2]  [1,2]  [3]  [1,3]  [2,3]  [1,2,3]
sums    :  0    1    2     3     3     4      5       6
```

### Approach A – build the subset, add it up at the end
Easy to read, but `sum(subset)` at every leaf costs O(n) extra per subset.

### Approach B – carry the running sum (better)
Pass `total` down the recursion: picking an element adds it to `total`, skipping leaves `total` unchanged. At the base case the answer is simply `total`. No list needed, so **O(2ⁿ)** time and only O(n) stack space.

In [1]:
class SubsetSumsA:
    def solve(self, nums, index, subset, result):
        if index >= len(nums):
            result.append(sum(subset))               # O(n) per leaf
            return
        subset.append(nums[index])                   # pick
        self.solve(nums, index + 1, subset, result)
        subset.pop()                                 # backtrack
        self.solve(nums, index + 1, subset, result)  # not pick

    def subset_sums(self, arr):
        result = []
        self.solve(arr, 0, [], result)
        result.sort()
        return result

print("A:", SubsetSumsA().subset_sums([1, 2, 3]))

A: [0, 1, 2, 3, 3, 4, 5, 6]


In [2]:
class SubsetSumsB:
    def solve(self, arr, index, total, result):
        if index >= len(arr):
            result.append(total)
            return
        self.solve(arr, index + 1, total + arr[index], result)   # pick: add to the running sum
        self.solve(arr, index + 1, total, result)                # not pick

    def subset_sums(self, arr):
        result = []
        self.solve(arr, 0, 0, result)
        result.sort()                                # the recursion order isn't sorted: [6, 3, 4, 1, 5, 2, 3, 0]
        return result

print("B:", SubsetSumsB().subset_sums([1, 2, 3]))
assert SubsetSumsA().subset_sums([5, 2, 1]) == SubsetSumsB().subset_sums([5, 2, 1])

B: [0, 1, 2, 3, 3, 4, 5, 6]


> 🧹 The earlier version of B forgot to sort the result (output `[6, 3, 4, 1, 5, 2, 3, 0]`) and named a variable `sum`, which hides Python's built-in `sum()`.

## 2. Subsets II: input with duplicates

`nums = [1, 2, 2]` → the subsets `[2]` (using the first 2) and `[2]` (using the second 2) are the **same** subset, so it must appear only once.

### Approach A – generate everything, filter repeats
Sort first so equal values sit together, then use the pick / not-pick tree and skip a subset that is **already in `result`**. It works, but `subset not in result` scans the whole list every time → slow.

### Approach B – skip duplicates while generating (optimal)
Use a *for-loop* style recursion. At each level, loop over the remaining elements `i = index … n−1` and add `nums[i]` to the subset. To avoid duplicates:

> If `i > index` and `nums[i] == nums[i-1]`, **skip** it.

Meaning: *within the same level, don't start a subset with the same value twice.* (The first copy at `i == index` is allowed, so `[2, 2]` can still be built by going one level deeper.)

Every recursive call records the current subset, because **every partial subset is itself a valid answer**.

In [3]:
class SubsetsWithDupA:
    def solve(self, nums, index, subset, result):
        if index >= len(nums):
            if subset not in result:                 # O(#results) scan each time
                result.append(subset.copy())
            return
        subset.append(nums[index])
        self.solve(nums, index + 1, subset, result)
        subset.pop()
        self.solve(nums, index + 1, subset, result)

    def subsets_with_dup(self, nums):
        result = []
        nums.sort()                                  # equal numbers must be neighbours
        self.solve(nums, 0, [], result)
        return result

print("A:", SubsetsWithDupA().subsets_with_dup([1, 2, 2]))

A: [[1, 2, 2], [1, 2], [1], [2, 2], [2], []]


In [4]:
class SubsetsWithDupB:
    def solve(self, nums, index, subset, result):
        result.append(subset.copy())                 # every partial subset is an answer
        for i in range(index, len(nums)):
            if i > index and nums[i] == nums[i - 1]: # same value already tried at this level
                continue
            subset.append(nums[i])
            self.solve(nums, i + 1, subset, result)  # move past the element we just used
            subset.pop()                             # backtrack

    def subsets_with_dup(self, nums):
        result = []
        self.solve(sorted(nums), 0, [], result)
        return result

print("B:", SubsetsWithDupB().subsets_with_dup([1, 2, 2]))

# Cross-check on a harder input: both must contain exactly the same set of subsets
data = [4, 4, 4, 1, 4]
a = SubsetsWithDupA().subsets_with_dup(data.copy())
b = SubsetsWithDupB().subsets_with_dup(data)
assert sorted(map(tuple, a)) == sorted(map(tuple, b))
print(len(b), "unique subsets for", data)

B: [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
10 unique subsets for [4, 4, 4, 1, 4]


## 3. Letter combinations of a phone number

On an old phone keypad each digit maps to letters (`2 → abc`, `3 → def`, …). Given `"23"`, return every possible letter string: `ad, ae, af, bd, be, bf, cd, ce, cf`.

This is a **"choose one of several options at each position"** problem, so we use a `for` loop over the letters of the current digit, instead of pick / not-pick:

```
index 0 (digit 2): a b c
index 1 (digit 3):   d e f   for each of the above
```

**Time:** O(4ⁿ · n) (a digit has at most 4 letters) · **Space:** O(n) recursion stack

> 🐞 **Edge case fixed:** for an empty input `""` the old code returned `['']`, but the expected answer is `[]` (no combinations).

In [5]:
class LetterCombinations:
    def solve(self, char_map, digits, index, subset, result):
        if index >= len(digits):
            result.append("".join(subset))
            return
        for ch in char_map[digits[index]]:           # try every letter for this digit
            subset.append(ch)
            self.solve(char_map, digits, index + 1, subset, result)
            subset.pop()                             # backtrack

    def letter_combinations(self, digits):
        if not digits:                               # empty input → no combinations
            return []
        char_map = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl",
                    "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}
        result = []
        self.solve(char_map, digits, 0, [], result)
        return result

lc = LetterCombinations()
print(lc.letter_combinations("23"))              # ['ad','ae','af','bd','be','bf','cd','ce','cf']
print(lc.letter_combinations(""))                # []
print(len(lc.letter_combinations("79")))         # 4 * 4 = 16

['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']
[]
16


---
### ✅ Key takeaways
- Carry a **running value** (like `total`) down the recursion instead of recomputing it at the leaves.
- **Duplicates:** sort first, then skip `nums[i] == nums[i-1]` when `i > index`. It is much faster than checking `subset not in result`.
- **One-of-many choices** (phone digits) → `for` loop in the recursion; **yes/no choices** → pick / not-pick.
- Think about edge cases (`""`, empty list) before submitting.